In [1]:

import pickle
import os
os.chdir(r"C:\Users\angel\Desktop\Proyectos_privados\PROYECTOS\EasyMoney_Capstone")
from scipy.stats import chi2_contingency
from itertools import combinations
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, OrdinalEncoder
from src.utils import *
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score, f1_score, recall_score, roc_curve, precision_score
from sklearn.model_selection import GridSearchCV
import json
import joblib
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn import set_config

pd.set_option('display.max_columns', None)

set_config(transform_output="pandas")
pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_columns", None)


## 1. Carga de data procesada , imputada y limpia

In [7]:
dev_df = pd.read_csv(
    "data/processed/output_data_enriquesida_muestra_limpia_develop.csv", sep="|", low_memory=False)
val_df = pd.read_csv(
    "data/processed/output_data_enriquesida_muestra_limpia_validacion.csv",  sep="|", low_memory=False)


In [8]:
dev_df_X, dev_df_y, val_df_X, val_df_y = import_dev_val(
    dev_df, val_df, target="sales_account")


## 2. Resalizar el split de train y test en develop

In [9]:
X_train, X_test, Y_train, Y_test = split_dev(dev_df_X, dev_df_y)


## 3. Eleccion del modelo

In [10]:
# Asegurarnos de que sea 1D y valores int
from sklearn.utils.class_weight import compute_class_weight
y_train_clean = np.array(Y_train).ravel().astype(int)


classes = np.array([0, 1])
weights = compute_class_weight(
    class_weight='balanced', classes=classes, y=y_train_clean)
print(weights)

class_weights = dict(zip(np.unique(Y_train), weights))

neg, pos = np.bincount(y_train_clean)
scale_pos_weight = neg / pos

# Para RandomForestClassifier
model = RandomForestClassifier(random_state=42, class_weight=class_weights)

# Para DecisionTreeClassifier
# model = DecisionTreeClassifier(random_state=42)

# Para XGBClassifier
# model =  XGBClassifier(
#         learning_rate=0.05,
#         subsample=0.7,
#         colsample_bytree=0.7,
#         gamma=1,
#         min_child_weight=5,
#         eval_metric="logloss",
#         random_state=42
#     )



[0.59340085 3.17663508]


## 4.Guardar algoritmo y sus parametros

In [12]:
name_model = model.__class__.__name__

with open(f"results/model/logaritmo/{name_model}.pkl", "wb") as f:
    pickle.dump(model, f)


------------------------------------------------------------------------------------------------